# Preparation for NN Training
### Read PAMTRA Simulations and corresponding 2D files
### save as Numpy

### LOAD AND PREPROCESS TRAINING DATA
CREATE NEW SET OF  TRAININGS DATA

#### Modules to load

In [3]:
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
from netCDF4 import Dataset
import seaborn as sns
from glob import glob
import xarray as xr
import pandas as pd
from functools import partial
##import typhon as ty
import sys
sys.path.append('/home/u/u301238/master_thesis/')
sys.path.append('/home/u/u301032/orcestra/NN_IWP_retrieval/NN_training_and_development/')
sys.path.append('/home/u/u301032/orcestra/NN_IWP_retrieval/')

#import src
import src_comparison_halo_pamtra as chp

#### Info for files

In [10]:
dates=["0824","0829","0927"]
appendices=["-rerun","-high3Drate","-rerun"]
name_pamtra_run =  "cells_025x025_2h" #"all_area_1000th_cell" #"cells_025x025_2h-test"#"all_area_1000th_cell" #
if name_pamtra_run == "all_area_1000th_cell":
    # Pamtra files
    time_selection = "4h"
    cell_selection = np.load('/home/u/u301032/orcestra/NN_IWP_retrieval/NN_training_and_development/cells_all_area_1000th_cell.npy')
    pamtra_files ='/work/um0203/u301032/PAMTRA_output/PAMTRA-ICON_*_all_area_v1.nc'
elif name_pamtra_run == "cells_025x025_2h":
    cell_selection = np.load('/home/u/u301032/orcestra/NN_IWP_retrieval/NN_training_and_development/cells_025x025_sea.npy')
    time_selection = "2h"
    pamtra_files ='/work/um0203/u301032/PAMTRA_output/PAMTRA-ICON_*_025x025_2h_v1.nc'
    print(pamtra_files)
elif name_pamtra_run == "cells_025x025_2h-test":
    cell_selection = np.load('/home/u/u301032/orcestra/NN_IWP_retrieval/NN_training_and_development/cells_025x025_sea.npy')
    time_selection = "2h"
    pamtra_files ='/work/um0203/u301032/PAMTRA_output/PAMTRA-ICON_0829*_025x025_2h_v1.nc'
    dates=dates[1]
    appendices=appendices[1]
    print(pamtra_files)
else:
    print('please insert correct name of pamtra run')
# Corresponding icon 2D files


altitude = 14450 #ORCESTRA  12000 #AC3 #Example heights #TODO: Automate for all flightlevels 

/work/um0203/u301032/PAMTRA_output/PAMTRA-ICON_*_025x025_2h_v1.nc


#### Load Trainingsdata

In [4]:
flight_levels =[13850,15000	]#13250,13600,11400,12650,13000,#14450,
for altitude in flight_levels:
    TBs = chp.load_nn_training_data_pamtra(pamtra_files,altitude=altitude)
    np.save('/work/um0203/u301032/master_thesis/ML_input/' + name_pamtra_run + '_TBs_altitude_' + str(altitude) + 'm.npy',TBs)
    


In [13]:
#TBs.shape
#IWV,IWP,LWP,qivi,qgvi,qsvi,cllvi,qrvi,t_steps = chp.load_nn_training_data_icon(dates,appendices,cell_selection,time_selection)
IWP= np.sum([qivi,qgvi,qsvi], axis=0)
LWP = np.sum([cllvi,qrvi], axis=0)
IWP

array([2.84269787e-02, 1.14829704e-01, 6.12308607e-02, ...,
       3.97277836e-06, 1.24780790e-05, 4.46089107e-05], dtype=float32)

### Load and write Trainings- and Testing data to numpy files

In [14]:
#np.save("TBs",TBs)

data=IWP,LWP,qivi,qgvi,qsvi,cllvi,qrvi #TBs, IWV,#,t_steps,cell_selection
names=[ 'IWP', 'LWP','qivi','qgvi','qsvi','cllvi','qrvi']#'TBs', 'IWV',#,'t_steps','cell_selection'
for i in range(len(names)):
    np.save('/work/um0203/u301032/master_thesis/ML_input/' + name_pamtra_run + '_' + names[i] + '.npy',data[i])


In [1]:
name_pamtra_run = "all_area_1000th_cell" # "cells_025x025_2h"
data=[]
names=['TBs', 'IWV', 'IWP', 'LWP','t_steps','cell_selection']
for i in range(len(names)):
    data.append(np.load('/work/um0203/u301032/master_thesis/ML_input/' + name_pamtra_run + '_' + names[i] + '.npy'))
TBs, IWV, IWP, LWP,t_steps,cell_selection =data

NameError: name 'np' is not defined

#### Filter for unrealistic BTs missing

In [ ]:
TB_input_vector=TBs
"""
# exclude profiles with unrealistic pamtra simulations
TBs[TBs[:,20]<230] = np.nan   # TODO what are in our case unrealistic values?
TB_input_vector = TBs[~np.isnan(TBs).any(axis=1),:] #TODO: count NANs. If NAN do not just drop , as index would get messed up
IWP = IWP[~np.isnan(TBs).any(axis=1)]
LWP = LWP[~np.isnan(TBs).any(axis=1)]
IWV = IWV[~np.isnan(TBs).any(axis=1)]
"""
if np.count_nonzero(np.isnan(TBs)) > 0:
    print("NaN values in Brightnesstemperature array. This may influence the rest of the Retrievaldeveloment, as they are not being filtered.")


In [ ]:
# set IWP values below 1gm2 to zero #TODO check what this means in tropics eg literature, histograms
#IWP[IWP<1.]=0.
#Das würde fast alle IWPs auf 0 setzen, da:
plt.plot(IWP,'x')
#check how different the subsets are:
x=np.arange(0,TB_input_vector.shape[0]+1,TB_input_vector.shape[0]/len(t_steps))
plt.plot(LWP,'x')
plt.plot(x,np.zeros_like(x),'o')
plt.show()
t_steps

In [ ]:
TBs.shape
# Only necessary if one wants to exclude certain frequencies.
TB_input_vector = np.concatenate((
        TB_input_vector[:,0:7], # K-Band
        TB_input_vector[:,7:14], # V-Band
        TB_input_vector[:,14:15], # W-Band
        TB_input_vector[:,15:19], # F-Band
        TB_input_vector[:,19:]), # G-Band
        axis=1)
TB_input_vector.shape

if (len(t_steps)*len(cell_selection)!=(TB_input_vector.shape[0])):
    print("Shape of TB does not match cell and time steps. Please check.")

len(t_steps)

In [ ]:
def splitting_in_train_test_validate(A,t_steps,slices_train,slices_validate,slices_test):    
    # Splits in timesteps
    x=np.arange(0,A.shape[0]+1,A.shape[0]/len(t_steps))
    x=x.astype(int)
    indexes = np.concatenate([np.arange(x[i],x[j]) for i,j in slices_train])
    train=  A[indexes]
    
    indexes = np.concatenate([np.arange(x[i],x[j]) for i,j in slices_validate])
    validate=  A[indexes]
    
    indexes = np.concatenate([np.arange(x[i],x[j]) for i,j in slices_test])
    test=  A[indexes]
    
    return train,test,validate

slices_train=[[0,2],[4,6],[8,10],[12,14]]
slices_validate=[[2,3],[7,8],[10,11]]
slices_test=[[3,4],[6,7],[11,12]]


train_TB, test_TB, validate_TB = splitting_in_train_test_validate(TB_input_vector,t_steps,slices_train,slices_validate,slices_test)
train_IWV, test_IWV, validate_IWV = splitting_in_train_test_validate(IWV,t_steps,slices_train,slices_validate,slices_test)
train_LWP, test_LWP, validate_LWP = splitting_in_train_test_validate(LWP,t_steps,slices_train,slices_validate,slices_test)
train_IWP, test_IWP, validate_IWP = splitting_in_train_test_validate(IWP,t_steps,slices_train,slices_validate,slices_test)

In [ ]:
def standardize_nn_training_data(TBs_train):
        
    TBs_centered = np.zeros(TBs_train.shape)
    mu_train = np.zeros(TBs_train.shape[1])
    sigma_train = np.zeros(TBs_train.shape[1])
    for channel in range(TBs_train.shape[1]):

        mu_train[channel] = np.nanmean(TBs_train[:,channel])
        sigma_train[channel] = np.nanstd(TBs_train[:,channel])   

        TBs_centered[:,channel] = (TBs_train[:,channel] - mu_train[channel])/sigma_train[channel]
        
    return TBs_centered, mu_train, sigma_train
    
def standardize_nn_input_data_v2(TBs, mu, sigma):
    
    TBs = np.asarray(TBs)
    # if single TB observation provided extend dims to 2
    if len(TBs.shape) == 1:
        TBs = TBs[np.newaxis,:]
    
    TBs_centered = np.zeros(TBs.shape)
    for channel in range(TBs.shape[1]):

        TBs_centered[:,channel] = (TBs[:,channel] - mu[channel])/sigma[channel]
    
    return TBs_centered
    

TBs_train_scaled, mu_train, sigma_train = standardize_nn_training_data(train_TB) 
TBs_test_scaled = standardize_nn_input_data_v2(test_TB,mu=mu_train,sigma=sigma_train) 
TBs_validate_scaled = standardize_nn_input_data_v2(validate_TB,mu=mu_train,sigma=sigma_train)

In [ ]:

    
# split training data into train and test subsets
#TBs_train, TBs_test, IWP_train, IWP_test  = src.split_nn_training_data(TB_input_vector,IWP,split_ratio=0.75)
##TBs_train, TBs_test, IWP_train, IWP_test, LWP_train, LWP_test, IWV_train, IWV_test = src.split_nn_training_data(TB_input_vector,IWP,LWP=LWP,IWV=IWV,split_ratio=0.75)
# standardize all TBs along their respective channel
# as a normal distribution with mean 0 and std of 1
TBs_train_scaled, mu_train, sigma_train = src.standardize_nn_training_data(TBs_train) #TODO write summary of function. Why is it needed? wht does it do? TODO Adapt function
TBs_test_scaled = src.standardize_nn_input_data_v2(test_TB,mu=mu_train,sigma=sigma_train) #TODO write summary of function. How does ist interact with the previous one?  TODO Adapt function